# Fine-Tuning Laya for Autonomous Browser Tasks (Kaggle GPU T4 ×2 DDP)

[![PyPI version](https://img.shields.io/pypi/v/laya.svg)](https://pypi.org/project/laya/)
[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Model-convaiinnovations%2Flaya-blue)](https://huggingface.co/convaiinnovations/laya)
[![Reference](https://img.shields.io/badge/GitHub-browser--use%2Fjev--ultrafast-black)](https://github.com/browser-use/jev-ultrafast)

This notebook fine-tunes **Laya** (, 421M params, ModernBERT-large backbone) for **autonomous browser control tasks** using a **dynamic, indexed action space** inspired by [browser-use/jev-ultrafast](https://github.com/browser-use/jev-ultrafast).

### Training Architecture:
* **Model**:  (421M ModernBERT encoder + non-autoregressive decision heads)
* **Hardware**: Kaggle 2× NVIDIA T4 GPUs via Distributed Data Parallel ()
* **Objective**: Reinforcement Learning with Calibrated Decisions (RLCD) over strictly proper scoring rules (Brier + spherical) + soft cross-entropy guidance
* **Calibration**: Post-training L-BFGS temperature optimization on held-out slice
* **Speed**: ~33 ms per forward pass (evaluating operation and candidate targets in a single round trip)

---

### ⚠️ Kaggle Settings:
* **Accelerator**: Select ****
* **Internet**: ****
* **Output**: Saved to 


## 1. Environment & Dual T4 GPU Verification

In [ ]:
!nvidia-smi
import os, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, "Please set Kaggle Accelerator to GPU T4 x2!"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Dual T4 GPUs ready for DDP distributed training.")


## 2. Install Dependencies

In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate
import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)


## 3. Data Preprocessing for Dynamic Indexed Action Space
We transform browser observations into indexed tables and speculative decision questions (, , , ).

In [ ]:
import os, json, torch
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

MODEL_ID = "convaiinnovations/laya"
print(f"Fetching tokenizer and config from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

cfg["max_len"] = 1024
cfg["head_max_len"] = 256

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]
    
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    
    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t], "target": target, "label": label}

# Load LocalLLaMA/typed-decisions baseline or browser tasks split
print("Loading dataset...")
ds_train = load_dataset("LocalLLaMA/typed-decisions", "all", split="train")
items = []
for row in ds_train:
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    for qid, q in questions.items():
        if qid in gold:
            it = build_training_item(state, q, gold[qid])
            if it: items.append(it)

print(f"Preprocessed {len(items)} training sequences.")
torch.save(items, "/kaggle/working/browser_train_items.pt")


## 4. DDP Training Script ()
Multi-GPU distributed RLCD training with GRPO logit perturbation, proper scoring rewards, and rolling checkpoints.

In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids, "attention_mask": att, "marker_pos": mpos, "marker_mask": mmask,
        "target": target, "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

def fit_one_temp(sel):
    if len(sel) < 10: return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()
    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)

    all_items = torch.load("/kaggle/working/browser_train_items.pt", weights_only=False)
    rng = random.Random(42)
    indices = list(range(len(all_items)))
    rng.shuffle(indices)
    n_calib = max(50, int(len(all_items) * 0.10))
    calib_items = [all_items[i] for i in indices[:n_calib]]
    train_items = [all_items[i] for i in indices[n_calib:]]
    my_items = train_items[rank::world_size]

    EPOCHS = 4
    MICRO_BATCH = 8
    GRAD_ACCUM = 4
    GROUP_SIZE = 4
    SIGMA_START = 0.4
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": 2.5e-5},
        {"params": head_params, "lr": 1.0e-4}
    ], weight_decay=0.01)
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    if rank == 0:
        print(f"Training {len(train_items)} items across {world_size} GPUs (Effective Batch: {MICRO_BATCH*world_size*GRAD_ACCUM})")
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches, accum_step = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * (epoch / max(1, EPOCHS - 1))
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk: continue
            batch = collate_train_batch(chunk, tok.pad_token_id)
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["marker_pos"].to(device), batch["marker_mask"].to(device), batch["qtype"].to(device))
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()
            
            scaler.scale(loss).backward()
            accum_step += 1
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Done in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
        dist.barrier()

    if rank == 0:
        print("Fitting calibration temperatures on held-out slice...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(cb["input_ids"].to(device), cb["attention_mask"].to(device), cb["marker_pos"].to(device), cb["marker_mask"].to(device), cb["qtype"].to(device))
                l_np = l_sub.float().cpu().numpy()
                for r_idx, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r_idx, :k], it["target"]))
        
        fitted_temps = [1.2, 1.2, 1.2]
        for qt in range(3):
            sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
            if sel: fitted_temps[qt] = fit_one_temp(sel)
        print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        
        os.makedirs(output_dir, exist_ok=True)
        save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-browser-agent"
        cfg["temperature"] = fitted_temps
        cfg.pop("temperature_by_options", None)
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Fine-tuned model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


## 5. Launch Multi-GPU Fine-Tuning with 
Executes across both T4 GPUs via PyTorch DDP (~4 to 8 minutes).

In [ ]:
OUTPUT_DIR = "/kaggle/working/laya_browser_agent"
cmd = f"torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py {model_dir} {OUTPUT_DIR}"
print("Running:", cmd)
!{cmd}


## 6. Official Head-to-Head Benchmark Evaluation against TypeSafe Jev

In [ ]:
import time, json
import numpy as np
import pandas as pd
from datasets import load_dataset
import laya
from laya.common import ece_score

print("Loading test split...")
ds_test = load_dataset("LocalLLaMA/typed-decisions", "all", split="test")
agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")

predictions = []
latencies = []
for row in ds_test:
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    t0 = time.perf_counter()
    pred = agent_ft.predict(state, questions)
    latencies.append((time.perf_counter() - t0) * 1000)
    predictions.append({"pred": pred, "gold": gold, "questions": questions})

accuracies, brier_scores, all_confs, all_corrects = [], [], [], []
for item in predictions:
    for qid, qdef in item["questions"].items():
        p_ans = item["pred"][qid]
        g_ans = item["gold"][qid]
        if qdef["type"] == "choice":
            is_corr = float(str(p_ans.get("choice", "")) == str(g_ans.get("label", "")))
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            keys = list(qdef["criteria"].keys())
            p_probs = np.array([p_ans["probabilities"].get(k, 1e-6) for k in keys])
            g_probs = np.array([g_ans["probabilities"].get(k, 1e-6) for k in keys])
            p_probs /= p_probs.sum()
            g_probs /= g_probs.sum()
            all_confs.append(float(p_probs.max()))
            brier_scores.append(float(((p_probs - g_probs) ** 2).sum()))

laya_acc = np.mean(accuracies)
laya_brier = np.mean(brier_scores)
laya_ece = ece_score(np.array(all_confs), np.array(all_corrects))
laya_lat = np.percentile(latencies, 50)

comparison = [
    {"Model": "TypeSafe Jev 1.13.0", "Kind": "Cloud API", "Accuracy": 0.727, "Brier": 0.148, "ECE": 0.144, "ms/step": 710, "Cost/Step": "zsh.0004"},
    {"Model": "Laya Browser Agent (2xT4)", "Kind": "Self-Hosted", "Accuracy": round(laya_acc, 3), "Brier": round(laya_brier, 3), "ECE": round(laya_ece, 3), "ms/step": round(laya_lat, 1), "Cost/Step": "zsh.00"},
    {"Model": "ModernBERT-base (149M)", "Kind": "Specialist", "Accuracy": 0.646, "Brier": 0.119, "ECE": 0.179, "ms/step": 349, "Cost/Step": "zsh.00"}
]
print(pd.DataFrame(comparison).to_markdown(index=False))
